<a href="https://colab.research.google.com/github/virwang/Fanshawe_DL_Pytorch26W/blob/clean-main/notebooks/Project_YunJiungWang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO-6147-(01)-26W Deep Learning with Pytorch

## Project:

**Student Name:** Yun-Jiung Wang

**Student Number:** 1256222

**Date:** March 30th, 2026

**Description:**

## Setup Env

In [1]:
!pip install datasets
!pip install gradio

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns
from torch.utils.data import random_split
from tqdm import tqdm

## Device

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [13]:
EPOCH=20
BATCH_SIZE = 64
LEARNING_RATE=0.001

##Load dataset (if run in local)

In [3]:
from datasets import load_dataset

dataset = load_dataset('ethz/food101')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

...

data/validation-00000-of-00003.parquet:   0%|          | 0.00/423M [00:00<?, ?B/s]

data/validation-00001-of-00003.parquet:   0%|          | 0.00/413M [00:00<?, ?B/s]

data/validation-00002-of-00003.parquet:   0%|          | 0.00/426M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/75750 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25250 [00:00<?, ? examples/s]

## Load Dataset (Colab)

In [10]:
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.Food101(
    root='./data',
    split='train',
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

num_classes = len(train_dataset.classes)
print(f"Load Food101 Successfully！Num of Classes: {num_classes}")

Load Food101 Successfully！Num of Classes: 101


## Build the model

In [12]:
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Train

In [ ]:
for epoch in range(EPOCHS):
    running_loss = 0.0
    correct = 0
    total = 0

    # 使用 tqdm 包裝你的 train_loader
    # desc: 進度條左側的文字說明
    # leave: 訓練完後是否保留進度條
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS}")

    for i, (images, labels) in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        #
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # update process ontine
        current_acc = 100. * correct / total
        pbar.set_postfix({
            'Loss': f"{loss.item():.4f}",
            'Acc': f"{current_acc:.2f}%"
        })

    avg_loss = running_loss / len(train_loader)
    print(f"==> Epoch {epoch+1} ends，ang Loss: {avg_loss:.4f}, final Acc: {100.*correct/total:.2f}%")

print("Every Epoch has done training！")